<a href="https://colab.research.google.com/github/Smyles019/html-login-form-detector/blob/main/notebooks/01_data_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
!git clone https://github.com/Smyles019/html-login-form-detector.git

fatal: destination path 'html-login-form-detector' already exists and is not an empty directory.


In [24]:
%cd html-login-form-detector

/content/html-login-form-detector


In [25]:
import pandas as pd
import re
import pickle
from collections import Counter
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer


In [26]:
# 1. Load and merge data files
file_path_1 = 'data/raw/Dataset 1/train_login_form.csv'
file_path_2 = 'data/raw/Dataset 1/train_no_form.csv'

df1 = pd.read_csv(file_path_1)
df2 = pd.read_csv(file_path_2)
df = pd.concat([df1, df2], ignore_index=True)
print(f"Total records loaded: {len(df)} (file1: {len(df1)}, file2: {len(df2)})")

Total records loaded: 1183 (file1: 502, file2: 681)


In [27]:
# 1-1. Checking & Handling & varifying missing values
n_missing = df['html_signature'].isna().sum()
if n_missing:
    print(f"⚠️ Found {n_missing} missing html_signature values → replacing with empty string")
df['html_signature'] = df['html_signature'].fillna('')

⚠️ Found 2 missing html_signature values → replacing with empty string


In [28]:
# 1-2. Check missing SHA-256 values

n_missing_sha = df['sha256'].isna().sum()

print(f"Missing sha256 values: {n_missing_sha}")

Missing sha256 values: 0


In [29]:
# 1-3. Remove duplicate pages using SHA-256

before = len(df)

df = (
    df.drop_duplicates(subset='sha256', keep='first')
      .reset_index(drop=True)
)

duplicates_removed = before - len(df)

print(f"Duplicate sha256 records removed: {duplicates_removed}")
print(f"Records remaining: {len(df)}")



Duplicate sha256 records removed: 0
Records remaining: 1183


In [30]:
# 1-4. Analyze repeated HTML signatures

signature_counts = df['html_signature'].value_counts()

repeated_signatures = signature_counts[signature_counts > 1]

print(f"Unique HTML signatures: {df['html_signature'].nunique()}")
print(f"Repeated HTML signatures: {len(repeated_signatures)}")
print(f"Records using repeated signatures: {repeated_signatures.sum()}")

Unique HTML signatures: 1141
Repeated HTML signatures: 34
Records using repeated signatures: 76


In [31]:
# Check whether HTML signatures have consistent labels

signature_labels = (
    df.groupby('html_signature')['label']
      .nunique()
)

conflicting_signatures = signature_labels[
    signature_labels > 1
]

print(
    f"HTML signatures appearing with multiple labels: "
    f"{len(conflicting_signatures)}"
)

HTML signatures appearing with multiple labels: 0


In [32]:
# 1-5. Analyze repeated HTML signatures

# Step 1 — Count how often each signature occurs

signature_counts = df['html_signature'].value_counts()

repeated_signatures = signature_counts[signature_counts > 1]

print("=" * 60)
print("REPEATED HTML SIGNATURE ANALYSIS")
print("=" * 60)

print(f"Total records: {len(df)}")
print(f"Unique HTML signatures: {df['html_signature'].nunique()}")
print(f"Repeated HTML signatures: {len(repeated_signatures)}")
print(f"Records belonging to repeated signatures: {repeated_signatures.sum()}")

REPEATED HTML SIGNATURE ANALYSIS
Total records: 1183
Unique HTML signatures: 1141
Repeated HTML signatures: 34
Records belonging to repeated signatures: 76


In [33]:
# Step 2 — See the most repeated signatures

print("\nTop 20 repeated HTML signatures:")

print(
    repeated_signatures
    .head(20)
)


Top 20 repeated HTML signatures:
html_signature
(body(div(div(div(div(div(span(span)(span(span))))(a(span))(div(div(div)(ul))))(div(div(div(div(form(div(ul(label(span))(div(div(label(span))(div(input))(div)))(div(div(label(span))(div(input))(div)))))(div(input))(div(input)(input)(input)(input)(input)(a(span))))(iframe)(div))))))))(div(div(div(div(ul)))))(div(div(style)(script)(script)))(script)(script)(script)(div)(script)(script)(script)(script)(script)(script))                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

In [34]:
# Step 3 — Check whether repeated signatures have different labels

signature_label_analysis = (
    df.groupby('html_signature')['label']
      .agg(
          samples='count',
          unique_labels='nunique'
      )
)

conflicting_signatures = signature_label_analysis[
    signature_label_analysis['unique_labels'] > 1
]

print("\n" + "=" * 60)
print("LABEL CONSISTENCY")
print("=" * 60)

print(
    f"Signatures appearing with multiple labels: "
    f"{len(conflicting_signatures)}"
)


LABEL CONSISTENCY
Signatures appearing with multiple labels: 0


In [35]:
# Step 4 — Inspect the conflicting signatures

print("\nConflicting signatures:")

print(
    conflicting_signatures
    .sort_values('samples', ascending=False)
    .head(20)
)


Conflicting signatures:
Empty DataFrame
Columns: [samples, unique_labels]
Index: []


In [36]:
# 1-6. Check label consistency of HTML signatures

signature_label_check = (
    df.groupby('html_signature')['label']
      .agg(
          sample_count='count',
          unique_labels='nunique'
      )
)

# Signatures associated with more than one label
conflicting_signatures = signature_label_check[
    signature_label_check['unique_labels'] > 1
]

print("=" * 60)
print("HTML SIGNATURE LABEL CONSISTENCY")
print("=" * 60)

print(f"Total unique HTML signatures: {len(signature_label_check)}")
print(f"Signatures with conflicting labels: {len(conflicting_signatures)}")

if len(conflicting_signatures) == 0:
    print("✓ All HTML signatures are label-consistent.")
else:
    print("⚠️ Some HTML signatures have multiple labels.")

HTML SIGNATURE LABEL CONSISTENCY
Total unique HTML signatures: 1141
Signatures with conflicting labels: 0
✓ All HTML signatures are label-consistent.


In [37]:
# Check label consistency among repeated signatures only

repeated_signature_labels = signature_label_check[
    signature_label_check['sample_count'] > 1
]

conflicting_repeated = repeated_signature_labels[
    repeated_signature_labels['unique_labels'] > 1
]

print("\nRepeated HTML signatures:", len(repeated_signature_labels))
print("Conflicting repeated signatures:", len(conflicting_repeated))


Repeated HTML signatures: 34
Conflicting repeated signatures: 0


In [38]:
# 2-1. Extract structural HTML features

import re
from collections import Counter

def extract_structural_features(sig):
    # Extract HTML tag names
    tags = re.findall(
        r'[a-zA-Z][a-zA-Z0-9]*',
        sig.lower()
    )

    tag_counts = Counter(tags)

    # Calculate maximum DOM depth
    max_depth = 0
    current_depth = 0

    for char in sig:
        if char == '(':
            current_depth += 1
            max_depth = max(max_depth, current_depth)
        elif char == ')':
            current_depth = max(0, current_depth - 1)

    form_count = tag_counts.get('form', 0)
    input_count = tag_counts.get('input', 0)

    return {
        'signature_length': len(sig),
        'max_depth': max_depth,
        'total_tags': len(tags),
        'unique_tags_count': len(tag_counts),

        # Important HTML tags
        'form_count': form_count,
        'input_count': input_count,
        'button_count': tag_counts.get('button', 0),
        'script_count': tag_counts.get('script', 0),
        'a_count': tag_counts.get('a', 0),
        'img_count': tag_counts.get('img', 0),
        'div_count': tag_counts.get('div', 0),

        # Relationship between inputs and forms
        'input_per_form_ratio': (
            input_count / form_count
            if form_count > 0 else 0.0
        )
    }

In [39]:
# Apply feature extraction to all HTML signatures

df_struct = pd.DataFrame(
    [
        extract_structural_features(sig)
        for sig in df['html_signature']
    ]
)

print("=" * 60)
print("STRUCTURAL HTML FEATURES")
print("=" * 60)

print(f"Feature matrix shape: {df_struct.shape}")

print("\nFeature names:")
print(df_struct.columns.tolist())

print("\nFirst 5 rows:")
print(df_struct.head())

STRUCTURAL HTML FEATURES
Feature matrix shape: (1183, 12)

Feature names:
['signature_length', 'max_depth', 'total_tags', 'unique_tags_count', 'form_count', 'input_count', 'button_count', 'script_count', 'a_count', 'img_count', 'div_count', 'input_per_form_ratio']

First 5 rows:
   signature_length  max_depth  total_tags  unique_tags_count  form_count  \
0               414          9          66                 18           1   
1              3607         13         823                 22           2   
2               392          9          74                 11           0   
3              8304         16        1608                 17           1   
4              1231         11         282                 22           1   

   input_count  button_count  script_count  a_count  img_count  div_count  \
0            1             4            10        2          0         10   
1            9            23             9      274          5         87   
2            1            

In [40]:
# 2-2. Validate structural features
import numpy as np

print("=" * 60)
print("STRUCTURAL FEATURE VALIDATION")
print("=" * 60)

print("\nMissing values:")
print(df_struct.isna().sum())

print("\nInfinite values:")
print(
    np.isinf(df_struct.select_dtypes(include='number')).sum()
)

print("\nFeature statistics:")
print(
    df_struct.describe().round(2)
)

STRUCTURAL FEATURE VALIDATION

Missing values:
signature_length        0
max_depth               0
total_tags              0
unique_tags_count       0
form_count              0
input_count             0
button_count            0
script_count            0
a_count                 0
img_count               0
div_count               0
input_per_form_ratio    0
dtype: int64

Infinite values:
signature_length        0
max_depth               0
total_tags              0
unique_tags_count       0
form_count              0
input_count             0
button_count            0
script_count            0
a_count                 0
img_count               0
div_count               0
input_per_form_ratio    0
dtype: int64

Feature statistics:
       signature_length  max_depth  total_tags  unique_tags_count  form_count  \
count           1183.00    1183.00     1183.00            1183.00     1183.00   
mean            2585.55      16.94      501.25              18.73        1.24   
std             4904.

In [41]:
# 3-1. Convert HTML signatures into tag sequences

def clean_signature(sig):
    """
    Convert HTML signature into a sequence of HTML tag tokens.
    """

    tags = re.findall(
        r'[a-zA-Z][a-zA-Z0-9]*',
        sig.lower()
    )

    return ' '.join(tags)


clean_signatures = df['html_signature'].apply(clean_signature)

print("=" * 60)
print("CLEANED HTML SIGNATURES")
print("=" * 60)

print(clean_signatures.head())

CLEANED HTML SIGNATURES
0    body div div header div svg title path div spa...
1    body style header div a a a nav div a form inp...
2    body nav div span img ul li li li div div div ...
3    body header nav div a img div div ul li a li a...
4    body div title meta meta meta link link script...
Name: html_signature, dtype: object


In [42]:
from sklearn.model_selection import train_test_split

X_struct_train, X_struct_test, y_train, y_test, sig_train, sig_test = (
    train_test_split(
        df_struct,
        df['label'],
        clean_signatures,
        test_size=0.20,
        random_state=42,
        stratify=df['label']
    )
)

print("=" * 60)
print("TRAIN / TEST SPLIT")
print("=" * 60)

print(f"Training samples: {len(X_struct_train)}")
print(f"Testing samples:  {len(X_struct_test)}")

TRAIN / TEST SPLIT
Training samples: 946
Testing samples:  237


In [43]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(
    ngram_range=(2, 3),
    token_pattern=r'\b[a-zA-Z][a-zA-Z0-9]*\b',
    min_df=2
)

In [44]:
# 3-2. Learn n-gram vocabulary from training data only

X_ngram_train = vectorizer.fit_transform(sig_train)

# Transform test data using the same vocabulary
X_ngram_test = vectorizer.transform(sig_test)

print("=" * 60)
print("HTML N-GRAM FEATURES")
print("=" * 60)

print(f"Training n-gram shape: {X_ngram_train.shape}")
print(f"Testing n-gram shape:  {X_ngram_test.shape}")

print(
    f"Number of learned n-grams: "
    f"{len(vectorizer.get_feature_names_out())}"
)

HTML N-GRAM FEATURES
Training n-gram shape: (946, 4893)
Testing n-gram shape:  (237, 4893)
Number of learned n-grams: 4893


In [45]:
# Step 10 - Inspect some learned HTML n-grams

feature_names = vectorizer.get_feature_names_out()

print("\nFirst 30 learned n-grams:")

for i, feature in enumerate(feature_names[:30], start=1):
    print(f"{i:2d}. {feature}")


First 30 learned n-grams:
 1. a a
 2. a a a
 3. a a article
 4. a a button
 5. a a code
 6. a a devsite
 7. a a div
 8. a a figure
 9. a a footer
10. a a form
11. a a header
12. a a hr
13. a a iframe
14. a a img
15. a a input
16. a a li
17. a a link
18. a a main
19. a a nav
20. a a ol
21. a a script
22. a a section
23. a a span
24. a a style
25. a a svg
26. a a table
27. a a td
28. a a time
29. a a tr
30. a a ul
